# 癫痫检测 MLP 模型结果展示

**模型**: 10 → 16 → 8 → 1, ReLU, Dropout 0.1, ~320 参数  
**数据**: CHB-MIT 24 病人,病人级 split (train 18人 / val 3人 / test 3人)  
**特征**: 10 个选定特征(时域+频域,来自 RF 置换重要性排序)  

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文显示
plt.rcParams['axes.unicode_minus'] = False

## 1. 训练曲线(健康性验证)

In [ ]:
curve = pd.read_csv('results/mlp/training_curve.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# 左图:训练 loss
ax1.plot(curve['epoch'], curve['train_loss'], marker='o', markersize=3)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train Loss (BCEWithLogits)')
ax1.set_title('训练损失曲线')
ax1.grid(alpha=0.3)

# 右图:val PR-AUC(early stopping 目标)
ax2.plot(curve['epoch'], curve['val_pr_auc'], marker='o', markersize=3, color='green')
best_ep = curve['val_pr_auc'].idxmax()
ax2.axvline(best_ep, color='red', linestyle='--', alpha=0.5, label=f'Best (ep{best_ep})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Val PR-AUC (real windows)')
ax2.set_title('验证集 PR-AUC (early stopping 指标)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"峰值: epoch {best_ep}, val PR-AUC = {curve.loc[best_ep, 'val_pr_auc']:.4f}")

## 2. pos_weight 选择(多 seed 稳定性)

In [ ]:
report = json.load(open('results/mlp/train_report.json'))
sweep = report['pos_weight_sweep']

ws = []
means = []
stds = []
for w_str, metrics in sweep.items():
    ws.append(float(w_str))
    means.append(metrics['val_pr_auc_mean'])
    stds.append(metrics['val_pr_auc_std'])

plt.figure(figsize=(8, 5))
plt.errorbar(ws, means, yerr=stds, marker='o', capsize=5, capthick=2)
plt.xlabel('pos_weight')
plt.ylabel('Val PR-AUC (mean ± std over 3 seeds)')
plt.title('pos_weight 选择(多 seed 验证)')
plt.grid(alpha=0.3)
best_w = report['pos_weight_chosen']
plt.axvline(best_w, color='red', linestyle='--', alpha=0.5, label=f'Chosen: {best_w}')
plt.legend()
plt.show()

print(f"选定 pos_weight = {best_w}")
print(f"Val PR-AUC = {sweep[str(best_w)]['val_pr_auc_mean']:.4f} ± {sweep[str(best_w)]['val_pr_auc_std']:.4f}")

## 3. Test 按病人拆分(关键结果)

In [ ]:
test_result = json.load(open('results/mlp/test_per_patient.json'))

# 转成 DataFrame
df = pd.DataFrame(test_result['test_per_patient'])
df = df[['patient', 'n_windows', 'n_positive', 'roc_auc', 'sensitivity', 'specificity', 'pr_auc']]
df.columns = ['病人', '窗口数', '阳性窗口', 'ROC-AUC', 'Sensitivity', 'Specificity', 'PR-AUC']

print("\n===== Test 按病人拆分 (threshold=0.75) =====")
print(df.to_string(index=False))
print(f"\n整体: ROC={test_result['test_overall']['roc_auc']:.3f}, "
      f"Sens={test_result['test_overall']['sensitivity']:.3f}, "
      f"Spec={test_result['test_overall']['specificity']:.3f}")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['ROC-AUC', 'Sensitivity', 'Specificity']
for i, m in enumerate(metrics):
    ax = axes[i]
    colors = ['red' if p=='chb06' else 'green' if p=='chb19' else 'orange' 
              for p in df['病人']]
    ax.bar(df['病人'], df[m], color=colors, alpha=0.7)
    ax.set_ylabel(m)
    ax.set_title(f'{m} (按病人)')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    
    # 标注数值
    for j, (p, v) in enumerate(zip(df['病人'], df[m])):
        ax.text(j, v+0.03, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 4. 关键结论

✅ **模型能力**: chb19 ROC=0.987,证明模型在"合适的病人"上非常强  
⚠️ **泛化局限**: chb06 ROC=0.568 接近随机,跨病人泛化不稳定  
📊 **整体**: 3 个 test 病人 1 好 1 中 1 差,是真实的病人异质性

**技术细节**:
- 训练曲线健康(lr=5e-5 修复)
- pos_weight=100(3 seeds 验证,std=0.008)
- threshold=0.75(窗口级 F1 峰值)
- 320 参数,硬件友好